Q1 — Medium
You have a DataFrame orders with columns:
order_id | customer_id | order_date | amount
Find the top 2 customers by total spend, but only considering orders placed in 2024. Return customer_id and total_spend, sorted descending.

In [0]:
# assume orders_df
import pyspark.sql.function as F 

final_df = (
  orders_df
  #.filter(expr("""order_date like '2024%'"""))
  .filter(F.year(F.col('order_date')) == 2024)
  .groupBy('customer_id')
  .agg(
    sum('amount').alias('total_spend')
  )
  #.select('customer_id', 'total_spend')
  #.sortBy(col('total_spend').desc())
  #.limit(2)
  .withColumn('amt_rank', dense_rank().over(Window.orderBy(F.col('total_spend').desc())))
  .filter(F.col('amt_rank') <= 2)
)

Q2 — Medium
You have two DataFrames:
employees: emp_id | name | dept_id | salary
departments: dept_id | dept_name | location
Find the average salary per department, but only for departments located in 'London'. Return dept_name and avg_salary, rounded to 2 decimal places.

In [0]:
# employees: emp_id | name | dept_id | salary
# departments: dept_id | dept_name | location

final_df = (
    employees
    .join(
            departments.filter(F.col('location') == 'London'), 
            on = 'dept_id', 
            how = 'inner'
        )
    .groupBy('dept_name')
    .agg(
        round(avg('salary'), 2).alias('avg_salary')
    )
)

Q3 — Medium-Hard
You have a DataFrame transactions:
user_id | txn_id | txn_date | amount
For each user, find the running total of amount ordered by txn_date. Also add a column txn_rank which ranks each transaction per user by date (earliest = rank 1).
Return all original columns plus running_total and txn_rank.

In [0]:
# transactions: user_id | txn_id | txn_date | amount

window = Window.partitionBy(F.col('user_id')).orderBy(F.col('txn_date'))

final_df = (
    transactions
    .withColumn('running_total', F.sum('amount').over(window))
    .withColumn('txn_rank', F.dense_rank().over(window))
)

Q4 — Medium-Hard
You have a DataFrame sessions:
user_id | session_id | session_start | session_end
session_start and session_end are timestamps.
For each user, find the longest gap in days between the end of one session and the start of the next session. Return user_id and max_gap_days.

In [0]:
# sessions: user_id | session_id | session_start | session_end

window = Window.partitionBy(F.col('user_id')).orderBy(F.col('session_start'))

final_df = (
    sessions
    .withColumn('gap_in_days', 
                    F.date_diff(
                        F.col('session_start'), 
                        F.lag(F.col('session_end'), 1).over(window)
                    )
                )
    # .withColumn('gap_rank', 
    #                 F.row_number().over(Window.partitionBy(F.col('user_id')).orderBy(F.col('gap_in_days').desc()))
    #             )
    # .filter(F.col('gap_rank') == 1)
    .filter(F.col('gap_in_days').isNotNull())
    .groupBy(F.col('user_id'))
    .agg(F.max(F.col('gap_in_days')).alias('max_gap_days'))
    # .select(F.col('user_id'), F.col('gap_in_days').alias('max_gap_days'))
)